In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

from agent_01 import save_company_pdf, ask_company
from agent_02 import plan_creative_strategy
from agent_03 import write_creative_scripts
from agent_04 import create_poster
from agent_05 import send_creative_email

c:\Users\Kaweesh\Desktop\Project\creative-agency-agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VIDEO EDITOR AGENT : Creating poster for company: Blue Bottle Cafe
PRODUCT MANAGER AGENT: sending creative for Blue Bottle Cafe to kaweeshravishanka8@gmail.com
Email sent to kaweeshravishanka8@gmail.com.

HOOK: Your new summer ritual is officially here ☀️🧊  
CAPTION: Sip The Summer with Blue Bottle Cafe’s new iced menu—cold, bright, and made fresh. 🧋✨ Stop by today and try your favorite iced pour!  
#BlueBottleCafe #SipTheSummer #IcedCoffee #FreshMade #SummerVibes #CafeSeason #CoffeeLovers #CondensationGoals


In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
openai = OpenAI(api_key=openai_api_key)
MODEL = "gpt-5.4-nano"

In [3]:
system_message = """
You are the Planner of a creative digital marketing agency.
You manage a team of agents. Each agent is one of your tools:
- save_company_pdf (RAG Agent): saves a client's company PDF into our database and analyzes it.
- ask_company_knowledge (RAG Agent): answers questions about a saved client.
- plan_creative_strategy (Strategy Agent): researches what's trending and plans the best poster for the client.
- write_creative_scripts (Creative Writer Agent): writes 1 poster script and 1 video script.
- create_poster (Video Editor Agent): designs the poster image.
- send_to_client (Product Manager Agent): writes hook + caption and emails the creative.

How to work:
1. When the user uploads a PDF, ALWAYS call save_company_pdf first, before anything else.
   If you don't know the company name, ask for it, then call save_company_pdf with that PDF.
2. When the user asks for a creative / poster / campaign, run the agents in this exact order:
   plan_creative_strategy -> write_creative_scripts -> create_poster -> send_to_client
3. If a tool says there is no data for the company, ask the user to upload their company PDF.
Keep replies to the user short: tell them what each agent did.
"""

In [4]:
save_pdf_function = {
    "name": "save_company_pdf",
    "description": "RAG Agent: save a client's company PDF into the RAG database and return a short company profile.",
    "parameters": {
        "type": "object",
        "properties": {
            "pdf_path": {"type": "string", "description": "Path of the uploaded PDF file"},
            "company_name": {"type": "string", "description": "Name of the client company"},
        },
        "required": ["pdf_path", "company_name"],
        "additionalProperties": False
    }
}

ask_company_function = {
    "name": "ask_company_knowledge",
    "description": "RAG Agent: answer a question about a client company using its saved documents.",
    "parameters": {
        "type": "object",
        "properties": {
            "company_name": {"type": "string", "description": "Name of the client company"},
            "question": {"type": "string", "description": "The question about the company"},
        },
        "required": ["company_name", "question"],
        "additionalProperties": False
    }
}

strategy_function = {
    "name": "plan_creative_strategy",
    "description": "Strategy Agent: search what's trending now and plan the best poster creative for the client.",
    "parameters": {
        "type": "object",
        "properties": {
            "company_name": {"type": "string", "description": "Name of the client company"},
            "goal": {"type": "string", "description": "What the client wants, e.g. 'promote our new summer menu'"},
        },
        "required": ["company_name", "goal"],
        "additionalProperties": False
    }
}

writer_function = {
    "name": "write_creative_scripts",
    "description": "Creative Writer Agent: write 1 poster script and 1 video script from the latest strategy.",
    "parameters": {"type": "object", "properties": {}, "additionalProperties": False}
}

poster_function = {
    "name": "create_poster",
    "description": "Video Editor Agent: design the poster image from the latest poster script.",
    "parameters": {"type": "object", "properties": {}, "additionalProperties": False}
}

send_function = {
    "name": "send_to_client",
    "description": "Product Manager Agent: write hook + caption and email the poster and scripts.",
    "parameters": {"type": "object", "properties": {}, "additionalProperties": False}
}

In [5]:
tools = [
    {"type": "function", "function": save_pdf_function},
    {"type": "function", "function": ask_company_function},
    {"type": "function", "function": strategy_function},
    {"type": "function", "function": writer_function},
    {"type": "function", "function": poster_function},
    {"type": "function", "function": send_function},
]

In [6]:
campaign = {}

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"PLANNER -> {name} {arguments}", flush=True)
        try:
            if name == "save_company_pdf":
                result = save_company_pdf(arguments["pdf_path"], arguments["company_name"])
            elif name == "ask_company_knowledge":
                result = ask_company(arguments["company_name"], arguments["question"])
            elif name == "plan_creative_strategy":
                campaign["company_name"] = arguments["company_name"]
                campaign["strategy"] = plan_creative_strategy(arguments["company_name"], arguments["goal"])
                result = campaign["strategy"]
            elif name == "write_creative_scripts":
                campaign["scripts"] = write_creative_scripts(campaign["company_name"], campaign["strategy"])
                result = campaign["scripts"]
            elif name == "create_poster":
                campaign["poster_path"] = create_poster(campaign["company_name"], campaign["scripts"])
                result = f"Poster saved to {campaign['poster_path']}"
            elif name == "send_to_client":
                result = send_creative_email(campaign["company_name"], campaign["scripts"], campaign["poster_path"])
            else:
                result = f"Unknown tool {name}"
        except Exception as e:
            result = f"Error in {name}: {e}"  # the planner sees the error and can fix it / ask the user
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses

In [7]:
def text_of(content):
    # Gradio gives content as a list like [{"type": "text", "text": "..."}, {"type": "file", "file": {"path": ...}}]
    # We keep the file paths too, so the Planner still knows about a PDF uploaded in an earlier message
    if isinstance(content, str):
        return content
    parts = []
    for c in content:
        if c.get("type") == "text":
            parts.append(c["text"])
        elif c.get("type") == "file":
            parts.append(f"[User uploaded a PDF: {c['file']['path']}]")
    return "\n".join(parts)

def chat(message, history):
    user_text = message["text"]
    for file in message["files"]:
        user_text += f"\n[User uploaded a PDF: {file}]"

    history = [{"role": h["role"], "content": text_of(h["content"])} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": user_text}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools, parallel_tool_calls=False)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools, parallel_tool_calls=False)

    return response.choices[0].message.content

In [8]:
gr.ChatInterface(
    fn=chat,
    multimodal=True,
    textbox=gr.MultimodalTextbox(file_types=[".pdf"], placeholder="Upload your company PDF or ask for a creative..."),
    title="Creative Digital Marketing Agency",
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
